# YOLO11m High-Resolution Teacher Candidate; BDD100K 10K Detection Subset

This notebook trains a YOLO11m detector on the `train_010k_seed42` BD100K detection subset and evaluates on the full `val_full` split at higher resolution.

Purpose is to test whether a larger, higher resolution YOLO model improves small object detection enough to serve as a future KD teacher.

```yaml
experiment:
    task: detection
    model: yolo11m.pt
    dataset: bdd100k-det
    train_split: train_010k_seed42
    val_split: val_full
    image_size: 960
    rectangular_batches: true
    cosine_lr: true
    epochs: 50
    save_dir: MyDrive/projects/ane-drive-perc/runs/teachers/yolo11m_bdd010k_rect960_cos50
```


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1S8wAO3470W9Nbg5CL0YWcMa4jwdMBmUZ?usp=sharing)

In [1]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.environ["REPO_URL"] = "https://github.com/aekn/ane-drive-perc.git"
os.environ["BRANCH"] = "main"
os.environ["HF_REPO_ID"] = "aekn/ane-drive-perc-bdd100k"

os.environ["PROJECT_ROOT"] = "/content/ane-drive-perc"
os.environ["GDRIVE_PROJECT_ROOT"] = "/content/drive/MyDrive/projects/ane-drive-perc"
os.environ["RUNS_DIR"] = "/content/drive/MyDrive/projects/ane-drive-perc/runs"
os.environ["MPLBACKEND"] = "Agg"

print("REPO_URL:", os.environ["REPO_URL"])
print("BRANCH:", os.environ["BRANCH"])
print("PROJECT_ROOT:", os.environ["PROJECT_ROOT"])
print("RUNS_DIR:", os.environ["RUNS_DIR"])

Mounted at /content/drive
REPO_URL: https://github.com/aekn/ane-drive-perc.git
BRANCH: main
PROJECT_ROOT: /content/ane-drive-perc
RUNS_DIR: /content/drive/MyDrive/projects/ane-drive-perc/runs


In [2]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 111.4 MB/s eta 0:00:00


In [3]:
!rm -rf "$PROJECT_ROOT"
!git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_ROOT"

%cd /content/ane-drive-perc

!mkdir -p "$RUNS_DIR"
!uv sync

Cloning into '/content/ane-drive-perc'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 124 (delta 33), reused 112 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 176.42 KiB | 1.39 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/ane-drive-perc
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 71 packages in 1ms
⠙ Preparing packages... (0/70)                                                  warning: `build_system.requires = ["uv-build>=0.9.21,<0.10.0"]` does not contain the current uv version 0.11.8
Prepared 70 packages in 36.53s
Installed 70 packages in 255ms
 + ane-drive-perc==0.1.0 (from file:///content/ane-drive-perc)
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4


In [4]:
%cd /content/ane-drive-perc

!nvidia-smi

/content/ane-drive-perc
Mon May  4 09:27:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

In [5]:
%cd /content/ane-drive-perc

import subprocess

train_proc = subprocess.Popen([
    "uv", "run", "python", "scripts/data/pull_bdd_shards.py",
    "--repo-id", os.environ["HF_REPO_ID"],
    "--split", "train_010k_seed42",
    "--out-dir", "data/materialized/bdd100k-det",
    "--token", os.environ["HF_TOKEN"],
])

val_proc = subprocess.Popen([
    "uv", "run", "python", "scripts/data/pull_bdd_shards.py",
    "--repo-id", os.environ["HF_REPO_ID"],
    "--split", "val_full",
    "--out-dir", "data/materialized/bdd100k-det",
    "--token", os.environ["HF_TOKEN"],
])

train_proc.wait()
val_proc.wait()

/content/ane-drive-perc


0

In [6]:
%cd /content/ane-drive-perc

!uv run python scripts/data/export_bdd_yolo.py \
  --train-split-dir data/materialized/bdd100k-det/train_010k_seed42 \
  --val-split-dir data/materialized/bdd100k-det/val_full \
  --out-dir data/yolo/bdd100k-det \
  --yaml-name bdd100k_det_010k.yaml

/content/ane-drive-perc
[train] train_010k_seed42: {'images': 10000, 'boxes': 184597, 'dropped': 52}
[val]   val_full: {'images': 10000, 'boxes': 185526, 'dropped': 52}
[done]  wrote data/yolo/bdd100k-det/bdd100k_det_010k.yaml


In [7]:
%cd /content/ane-drive-perc

!find data/yolo/bdd100k-det -maxdepth 3 -type d | sort
!echo ""
!cat data/yolo/bdd100k-det/bdd100k_det_010k.yaml
!echo ""
!echo "train labels:"
!find data/yolo/bdd100k-det/labels/train_010k_seed42 -name "*.txt" | wc -l
!echo "train images/symlinks:"
!find data/yolo/bdd100k-det/images/train_010k_seed42 \( -name "*.jpg" -o -type l \) | wc -l

/content/ane-drive-perc
data/yolo/bdd100k-det
data/yolo/bdd100k-det/images
data/yolo/bdd100k-det/images/train_010k_seed42
data/yolo/bdd100k-det/images/val_full
data/yolo/bdd100k-det/labels
data/yolo/bdd100k-det/labels/train_010k_seed42
data/yolo/bdd100k-det/labels/val_full

path: /content/ane-drive-perc/data/yolo/bdd100k-det
train: images/train_010k_seed42
val: images/val_full

nc: 10
names:
  0: pedestrian
  1: rider
  2: car
  3: truck
  4: bus
  5: train
  6: motorcycle
  7: bicycle
  8: traffic light
  9: traffic sign

train labels:
10000
train images/symlinks:
10000


In [8]:
os.environ["MPLBACKEND"] = "Agg"

In [9]:
%cd /content/ane-drive-perc

!uv run yolo detect train \
  model=yolo11m.pt \
  data=data/yolo/bdd100k-det/bdd100k_det_010k.yaml \
  imgsz=960 \
  rect=True \
  epochs=50 \
  batch=24 \
  device=0 \
  workers=8 \
  seed=42 \
  cos_lr=True \
  project="$RUNS_DIR/teachers" \
  name=yolo11m_bdd010k_rect960_cos50

/content/ane-drive-perc
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data/yolo/bdd100k-det/bdd100k_det_010k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, h